## Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [16]:
import os
from langchain_groq import ChatGroq

llm = ChatGroq(
    model='openai/gpt-oss-120b',
    api_key= os.getenv("GROQ_API_KEY")
)

llm.invoke("hi")

APIConnectionError: Connection error.

In [3]:
from tkinter.filedialog import Directory

from pydantic import BaseModel, Field 

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    director: str= Field(description="The director of the movie")
    rating: float=Field(description="The movies rating out of 10")
    


In [4]:
model_with_structure = llm.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000195EAA96550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000195EABF9ED0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'pa

In [5]:
## without structure

llm.invoke('provide details about the movie inception')

AIMessage(content='**Inception (2010) – Quick Reference Guide**\n\n| Item | Details |\n|------|---------|\n| **Title** | *Inception* |\n| **Release Year** | 2010 |\n| **Genre** | Science‑fiction / Action / Thriller |\n| **Running Time** | 148 minutes (theatrical cut) |\n| **Country** | United States |\n| **Language** | Primarily English (with brief French, Japanese, and other dialogue) |\n| **Budget** | ≈\u202f$160\u202fmillion (production) |\n| **Box‑Office Gross** | ≈\u202f$836\u202fmillion worldwide (making it the 4th‑highest‑grossing film of 2010) |\n| **MPAA Rating** | PG‑13 (Violence, some language, brief drug use) |\n\n---\n\n## 1. Creative Team\n\n| Role | Person |\n|------|--------|\n| **Director / Writer** | **Christopher Nolan** |\n| **Producer(s)** | Emma Thomas, Christopher Nolan, Charles Roven, Jordan Goldberg |\n| **Cinematographer** | Wally Pfister (Academy Award winner for this film) |\n| **Production Designer** | Guy Hendrix Dyas |\n| **Editor** | Lee Smith |\n| **Com

In [ ]:
## with schema

response = model_with_structure.invoke('provide details about the movie inception')

In [10]:
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

The ... (Ellipsis) tells Pydantic:

"This field is required."

In [11]:
## get raw message also with the structured message

from tkinter.filedialog import Directory

from pydantic import BaseModel, Field 

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="This year the movie was released")
    director: str= Field(..., description="The director of the movie")
    rating: float=Field(..., description="The movies rating out of 10")
    
model_with_structure = llm.with_structured_output(Movie, include_raw=True)

response = llm.invoke('provide details about the movie inception')

response

AIMessage(content='**Inception (2010) – Overview & Key Details**\n\n| Category | Information |\n|----------|--------------|\n| **Title** | *Inception* |\n| **Release Date** | July\u202f16,\u202f2010 (United States) |\n| **Genre** | Science‑fiction, Action, Thriller |\n| **Running Time** | 148 minutes |\n| **Director** | Christopher Nolan |\n| **Screenplay** | Christopher Nolan (story & script) |\n| **Producers** | Emma Thomas, Christopher Nolan, and others (including Jordan Goldberg & Christopher Nolan) |\n| **Production Companies** | Warner Bros. Pictures, Legendary Pictures, Syncopy Inc. |\n| **Budget** | Approximately **$160\u202fmillion** |\n| **Box‑Office Gross** | Roughly **$836\u202fmillion** worldwide |\n| **MPAA Rating** | PG‑13 (for sequences of violence and action, some language, and brief nudity) |\n\n---\n\n### Plot Summary (Spoiler‑Free)\n\n*Inception* follows **Dom Cobb** (Leonardo DiCaprio), a skilled “extractor” who enters people’s dreams to steal hidden secrets. He is

## Nested Structure

In [17]:
class Actor(BaseModel):
    name: str 
    role: str

class MovieDetails(BaseModel):
    title: str 
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description='Budget in USD')


output_with_structure = llm.with_structured_output(MovieDetails)

output_with_structure.invoke("provide details of movie inception")



MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=160000000.0)

## TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.